# Assignment 01 — Part-of-Speech (POS) Extraction from a News Article

**Course:** AIAC 536 — Natural Language Processing
**Programme:** MTech in Artificial Intelligence, Kathmandu University
**Student:** Prasanna Koirala

---

## What this assignment asks for

1. Take an English news article from a reputable source and save it as a `.txt` file
2. Load that text file into Python
3. Perform **Part-of-Speech (POS) tagging** on it
4. Extract **all the nouns and verbs**
5. Save them into a CSV file

## What is Part-of-Speech tagging?

Every word in a sentence plays a **grammatical role**. "Telescope" is a thing (a noun).
"Launched" is an action (a verb). "Powerful" describes something (an adjective).

**POS tagging is the job of automatically labelling every word with its role.**

Take this sentence:

| NASA | launched | a | powerful | telescope |
|---|---|---|---|---|
| PROPN | VERB | DET | ADJ | NOUN |
| *name* | *action* | *article* | *describes* | *thing* |

It sounds simple, but it is genuinely hard, because **the same word can be different
parts of speech depending on context**:

- "They **launch** the rocket" → *launch* is a **verb**
- "The **launch** was successful" → *launch* is a **noun**

A tagger has to read the surrounding words to decide. That is why we use a trained
statistical model rather than a dictionary lookup.

### Why does anyone care?

POS tags are a foundation that other NLP tasks are built on top of — finding names
(which is Assignment 02), pulling out what a sentence is *about*, translation, and
grammar checking all lean on knowing each word's role.

## Step 0 — Setup

We import the libraries and load the language model.

In [1]:
# --- Python's own built-in tools ---
from collections import Counter   # counts how many times each item appears in a list
from pathlib import Path          # a safe, readable way to handle file paths

# --- Third-party libraries ---
import pandas as pd               # builds the results table and writes the CSV
import spacy                      # the NLP library that performs the POS tagging

# spacy.explain() prints a harmless warning for punctuation tags like "(" that
# have no glossary entry. Hide it so the output stays readable.
import warnings
warnings.filterwarnings("ignore", message=r".*W118.*")

# Load spaCy's small English model.
#
# The name "en_core_web_sm" breaks down as:
#   en   = English
#   core = general-purpose (tagging, parsing, named entities)
#   web  = trained on text scraped from the web (blogs, news, comments)
#   sm   = small version (fast, ~12 MB; there are md/lg/trf versions that are
#          more accurate but much bigger)
#
# `nlp` is a pipeline object. You feed it text and it hands back an analysed
# document with every word already tagged.
nlp = spacy.load("en_core_web_sm")

print("spaCy version :", spacy.__version__)
print("Model         :", nlp.meta["name"], "v" + nlp.meta["version"])
print("Pipeline steps:", nlp.pipe_names)

spaCy version : 3.8.16
Model         : core_web_sm v3.8.0
Pipeline steps: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


That `Pipeline steps` list is worth a look. When you pass text to `nlp`, it runs
through those components in order. The one we care about here is **`tagger`** — the
component that assigns the part-of-speech tags.

## Step 1 — See the tagger work on a single sentence

Before running anything on a 900-word article, let's tag **one short sentence** so we
can see exactly what comes back and check it by eye.

In [2]:
# Pass a string to `nlp` and it returns a Doc object - the analysed sentence.
demo = nlp("NASA launched a powerful telescope on Sunday.")

# A Doc can be looped over. Each item is a "token" - roughly one word or one
# punctuation mark. For each token we look at three attributes:
#
#   token.text  -> the word exactly as it appeared in the text
#   token.pos_  -> the COARSE part of speech       (NOUN, VERB, ADJ, ...)
#   token.tag_  -> the FINE-GRAINED tag            (NN, NNS, VBD, ...)
#
# The trailing underscore matters! `token.pos` (no underscore) gives a number,
# which is how spaCy stores it internally. `token.pos_` gives the readable string.

print(f"{'WORD':<12} {'COARSE':<8} {'FINE':<6} WHAT THE FINE TAG MEANS")
print("-" * 68)

for token in demo:
    # spacy.explain() turns a cryptic tag like "VBD" into "verb, past tense"
    meaning = spacy.explain(token.tag_)
    print(f"{token.text:<12} {token.pos_:<8} {token.tag_:<6} {meaning}")

WORD         COARSE   FINE   WHAT THE FINE TAG MEANS
--------------------------------------------------------------------
NASA         PROPN    NNP    noun, proper singular
launched     VERB     VBD    verb, past tense
a            DET      DT     determiner
powerful     ADJ      JJ     adjective (English), other noun-modifier (Chinese)
telescope    NOUN     NN     noun, singular or mass
on           ADP      IN     conjunction, subordinating or preposition
Sunday       PROPN    NNP    noun, proper singular
.            PUNCT    .      punctuation mark, sentence closer


Notice that spaCy gives us **two** labels per word, not one:

- **`pos_`** is the *coarse* tag. There are only about 17 of these, and they are the
  same across every language spaCy supports. `NOUN`, `VERB`, `ADJ`, and so on.
- **`tag_`** is the *fine-grained* tag. It is English-specific and much more detailed —
  it distinguishes `NN` (singular noun) from `NNS` (plural noun), and `VBD` (past tense)
  from `VBG` (the *-ing* form).

Both are useful, so **we will keep both** in our final CSV.

## Step 2 — Load the news article

### About the article

| | |
|---|---|
| **Title** | NASA's Dark Universe-Seeking Nancy Grace Roman Space Telescope Launches |
| **Source** | NASA — [nasa.gov](https://www.nasa.gov/news-release/nasas-dark-universe-seeking-nancy-grace-roman-space-telescope-launches/) |
| **Retrieved** | 1 September 2026 |

**Why this article?** Three reasons. It comes from a reputable primary source. NASA
press releases are public domain (US Government works), so reproducing the text in a
submitted assignment raises no copyright problem. And it is short enough — about 850
words — that we can actually read the tagger's output and check whether it is correct,
which is the whole point of a learning exercise.

Full details are recorded in `data/SOURCE.md`.

In [3]:
# Path to the saved article. Using Path() rather than a plain string means this
# works the same on macOS, Linux and Windows.
ARTICLE_PATH = Path("data/article.txt")

# Read the whole file into one string.
# encoding="utf-8" is important: the article contains curly apostrophes (in words
# like "NASA's"), which are not plain ASCII characters.
article_text = ARTICLE_PATH.read_text(encoding="utf-8")

# Some quick sanity checks so we know the file actually loaded properly.
print("File loaded :", ARTICLE_PATH)
print("Characters  :", len(article_text))
print("Words (rough, by splitting on spaces):", len(article_text.split()))
print("Paragraphs  :", len([p for p in article_text.split("\n") if p.strip()]))

File loaded : data/article.txt
Characters  : 5559
Words (rough, by splitting on spaces): 843
Paragraphs  : 14


In [4]:
# Print the first 400 characters so we can confirm the text looks right
# and is not, say, HTML tags or an error message.
print(article_text[:400], "...")

NASA's Dark Universe-Seeking Nancy Grace Roman Space Telescope Launches

Now on a three-month, million-mile journey to its final orbit, NASA's Nancy Grace Roman Space Telescope will soon reveal the universe's darkest secrets. The mission launched at 7:26 a.m. EDT Sunday aboard a SpaceX Falcon Heavy rocket from Launch Complex 39A at the agency's Kennedy Space Center in Florida.

Roman pairs a large ...


## Step 3 — Tokenisation: splitting text into words

Before anything can be tagged, the text has to be split into individual units. That
step is called **tokenisation**, and it is less obvious than it sounds.

Splitting on spaces is not good enough. Consider `"NASA's"`:

- Split on spaces → one token: `NASA's`
- What we actually want → two tokens: `NASA` and `'s`

They are two different things! `NASA` is a name, and `'s` is a possessive marker
meaning "belonging to". A proper tokeniser knows this. Punctuation gets separated too,
so `Sunday.` becomes `Sunday` and `.`

In [5]:
# A short example that shows the tricky cases side by side.
tokenisation_demo = nlp("NASA's telescope launched Sunday, and it didn't fail.")

print("Naive split on spaces:")
print("  ", "NASA's telescope launched Sunday, and it didn't fail.".split())

print("\nspaCy's tokeniser:")
print("  ", [token.text for token in tokenisation_demo])

Naive split on spaces:
   ["NASA's", 'telescope', 'launched', 'Sunday,', 'and', 'it', "didn't", 'fail.']

spaCy's tokeniser:
   ['NASA', "'s", 'telescope', 'launched', 'Sunday', ',', 'and', 'it', 'did', "n't", 'fail', '.']


Look at the difference: spaCy correctly separated `NASA` + `'s`, split the comma off
`Sunday`, and even split `didn't` into `did` + `n't`. Naive splitting got all three
wrong. **Every later step depends on getting this right.**

## Step 4 — Tag the whole article

Now we run the full article through the pipeline. One line does all the work.

In [6]:
# Running nlp() on the article text tokenises AND tags it in one pass.
doc = nlp(article_text)

# `doc.sents` is a generator of sentences, so we convert it to a list to count it.
sentences = list(doc.sents)

print("Total tokens    :", len(doc))
print("Total sentences :", len(sentences))
print()

# Show the first two sentences fully tagged, so we can spot-check the output.
print("First two sentences, tagged:")
print("=" * 68)
for i, sentence in enumerate(sentences[:2], start=1):
    print(f"\nSentence {i}: {sentence.text.strip()}\n")
    for token in sentence:
        # Skip whitespace tokens - they carry no grammatical information.
        if not token.is_space:
            print(f"   {token.text:<18} {token.pos_:<8} {token.tag_}")

Total tokens    : 1012
Total sentences : 37

First two sentences, tagged:

Sentence 1: NASA's Dark Universe-Seeking Nancy Grace Roman Space Telescope Launches

Now on a three-month, million-mile journey to its final orbit, NASA's Nancy Grace Roman Space Telescope will soon reveal the universe's darkest secrets.

   NASA               PROPN    NNP
   's                 PART     POS
   Dark               PROPN    NNP
   Universe           PROPN    NNP
   -                  PUNCT    HYPH
   Seeking            VERB     VBG
   Nancy              PROPN    NNP
   Grace              PROPN    NNP
   Roman              PROPN    NNP
   Space              PROPN    NNP
   Telescope          PROPN    NNP
   Launches           PROPN    NNPS
   Now                ADV      RB
   on                 ADP      IN
   a                  DET      DT
   three              NUM      CD
   -                  PUNCT    HYPH
   month              NOUN     NN
   ,                  PUNCT    ,
   million            NUM

## Step 5 — Understanding the two tagsets

This is the part that confuses people most, so it is worth slowing down.

### Coarse tags (Universal POS / UPOS)

About 17 broad categories, shared across all languages:

| Tag | Meaning | Example from our article |
|---|---|---|
| `NOUN` | common noun — an ordinary thing | telescope, mission, data |
| `PROPN` | **proper** noun — a *name* | NASA, Roman, Florida |
| `VERB` | main action verb | launched, explore, reveal |
| `AUX` | **auxiliary** ("helper") verb | will, is, has, could |
| `ADJ` | adjective | dark, crisp, infrared |
| `ADP` | preposition | in, at, from |
| `DET` | determiner | the, a, this |

### Fine-grained tags (Penn Treebank)

English-specific and far more detailed. The nouns and verbs split like this:

| Fine tag | Meaning | Rolls up to |
|---|---|---|
| `NN` | noun, singular | `NOUN` |
| `NNS` | noun, plural | `NOUN` |
| `NNP` | proper noun, singular | `PROPN` |
| `NNPS` | proper noun, plural | `PROPN` |
| `VB` | verb, base form | `VERB` |
| `VBD` | verb, past tense | `VERB` |
| `VBG` | verb, gerund / *-ing* | `VERB` |
| `VBN` | verb, past participle | `VERB` |
| `VBP` | verb, present, non-3rd-person | `VERB` |
| `VBZ` | verb, present, 3rd-person singular | `VERB` |
| `MD` | modal ("will", "could") | `AUX` |

So **many fine tags collapse into one coarse tag**. Let's confirm that on the real data.

In [7]:
# Count how often each coarse tag appears.
# Counter() takes a list and returns {item: how_many_times}.
coarse_counts = Counter(token.pos_ for token in doc if not token.is_space)

print("COARSE TAG COUNTS (whole article)")
print("-" * 40)
for tag, count in coarse_counts.most_common():
    # .ljust() pads the tag with spaces so the columns line up neatly
    print(f"  {tag:<8} {count:>4}   {spacy.explain(tag)}")

COARSE TAG COUNTS (whole article)
----------------------------------------
  NOUN      215   noun
  PROPN     159   proper noun
  PUNCT     112   punctuation
  ADP       111   adposition
  DET        80   determiner
  VERB       75   verb
  ADJ        73   adjective
  AUX        35   auxiliary
  PART       33   particle
  ADV        31   adverb
  PRON       27   pronoun
  CCONJ      26   coordinating conjunction
  NUM        18   numeral
  SCONJ       2   subordinating conjunction
  SYM         1   symbol


In [8]:
# Now show which FINE tags roll up into which COARSE tag,
# but only for the nouns and verbs, since those are what the assignment wants.

print("HOW FINE TAGS MAP TO COARSE TAGS")
print("=" * 58)

for coarse in ["NOUN", "PROPN", "VERB", "AUX"]:
    # Collect the fine tags belonging to this coarse category
    fine_tags = Counter(t.tag_ for t in doc if t.pos_ == coarse and not t.is_space)
    total = sum(fine_tags.values())

    print(f"\n{coarse}  (total {total})")
    for fine, count in fine_tags.most_common():
        print(f"    {fine:<6} {count:>4}   {spacy.explain(fine)}")

HOW FINE TAGS MAP TO COARSE TAGS

NOUN  (total 215)
    NN      157   noun, singular or mass
    NNS      58   noun, plural

PROPN  (total 159)
    NNP     156   noun, proper singular
    NNPS      3   noun, proper plural

VERB  (total 75)
    VB       35   verb, base form
    VBG      11   verb, gerund or present participle
    VBD       9   verb, past tense
    VBZ       9   verb, 3rd person singular present
    VBN       9   verb, past participle
    VBP       2   verb, non-3rd person singular present

AUX  (total 35)
    MD       22   verb, modal auxiliary
    VBZ       6   verb, 3rd person singular present
    VBP       4   verb, non-3rd person singular present
    VB        2   verb, base form
    VBN       1   verb, past participle


## Step 6 — Extracting the nouns and verbs (and a decision to make)

The assignment says "extract all Nouns and Verbs". That sounds unambiguous, but it
hides two real questions.

### Question 1 — Do proper nouns count as nouns?

spaCy labels `NASA` and `Florida` as `PROPN`, not `NOUN`. Strictly, `PROPN` is a
*separate* coarse tag. But grammatically a proper noun **is** a noun — it is the name of
a thing.

### Question 2 — Do auxiliary verbs count as verbs?

Words like "will", "is" and "could" are tagged `AUX`, not `VERB`. They are helper verbs
that support a main verb ("**will** launch"). Again, they are still verbs
grammatically.

### Our decision

**Include both, but label them**, adding a `POS_Tag` column that records the exact tag.
Anyone reading the CSV can filter to strict `NOUN`/`VERB` if they prefer — nothing is
lost. Excluding them silently would throw away real data.

The cell below shows exactly how much data that decision is worth.

In [9]:
# Count each of the four categories separately so we can see the impact.
n_noun  = sum(1 for t in doc if t.pos_ == "NOUN")
n_propn = sum(1 for t in doc if t.pos_ == "PROPN")
n_verb  = sum(1 for t in doc if t.pos_ == "VERB")
n_aux   = sum(1 for t in doc if t.pos_ == "AUX")

print("WHAT THE DECISION IS WORTH")
print("=" * 52)
print(f"  NOUN  (common nouns) : {n_noun:>4}")
print(f"  PROPN (proper nouns) : {n_propn:>4}")
print(f"  {'-' * 30}")
print(f"  Total nouns          : {n_noun + n_propn:>4}")
# Work out what share of the nouns we would lose by excluding proper nouns
print(f"  -> excluding PROPN would drop {n_propn / (n_noun + n_propn):.0%} of all nouns")

print()
print(f"  VERB  (main verbs)   : {n_verb:>4}")
print(f"  AUX   (helper verbs) : {n_aux:>4}")
print(f"  {'-' * 30}")
print(f"  Total verbs          : {n_verb + n_aux:>4}")
print(f"  -> excluding AUX would drop {n_aux / (n_verb + n_aux):.0%} of all verbs")

WHAT THE DECISION IS WORTH
  NOUN  (common nouns) :  215
  PROPN (proper nouns) :  159
  ------------------------------
  Total nouns          :  374
  -> excluding PROPN would drop 43% of all nouns

  VERB  (main verbs)   :   75
  AUX   (helper verbs) :   35
  ------------------------------
  Total verbs          :  110
  -> excluding AUX would drop 32% of all verbs


That is a large effect, and it is not an accident. This is an article about a **space
mission**, so it is dense with names: NASA, Roman, Falcon Heavy, Goddard, Florida,
Maryland. A different article — an opinion column, say — would have a much lower
proportion of proper nouns.

**This is the kind of thing you only notice by checking rather than assuming.**

## Step 7 — Building the results table

Now we walk through every token and keep the ones that are nouns or verbs.

In [10]:
# We will collect one dictionary per extracted word, then turn the whole
# list into a pandas DataFrame (a table).
rows = []

# Which coarse tags we are treating as nouns and as verbs (see Step 6).
NOUN_TAGS = {"NOUN", "PROPN"}
VERB_TAGS = {"VERB", "AUX"}

# enumerate(sentences, start=1) gives us (1, first_sentence), (2, second_sentence), ...
# so we can record which sentence each word came from.
for sentence_number, sentence in enumerate(sentences, start=1):
    for token in sentence:

        # Decide whether this token is one we want.
        if token.pos_ in NOUN_TAGS:
            category = "Noun"
        elif token.pos_ in VERB_TAGS:
            category = "Verb"
        else:
            continue          # not a noun or verb -> skip to the next token

        # Skip anything that is only whitespace or punctuation, just in case.
        if token.is_space or token.is_punct:
            continue

        rows.append({
            "Word":        token.text,            # the word as it appeared
            "POS_Tag":     token.pos_,            # coarse tag: NOUN/PROPN/VERB/AUX
            "Fine_Tag":    token.tag_,            # fine tag:   NN/NNS/VBD/...
            "Lemma":       token.lemma_,          # dictionary form: "launched" -> "launch"
            "Category":    category,              # our own grouping: Noun or Verb
            "Sentence_No": sentence_number,       # which sentence it came from
        })

# Turn the list of dictionaries into a table.
pos_df = pd.DataFrame(rows)

print("Rows extracted:", len(pos_df))
print()
pos_df.head(15)

Rows extracted: 483



,Word,POS_Tag,Fine_Tag,Lemma,Category,Sentence_No
0,NASA,PROPN,NNP,NASA,Noun,1
1,Dark,PROPN,NNP,Dark,Noun,1
2,Universe,PROPN,NNP,Universe,Noun,1
3,Seeking,VERB,VBG,seek,Verb,1
4,Nancy,PROPN,NNP,Nancy,Noun,1
5,Grace,PROPN,NNP,Grace,Noun,1
6,Roman,PROPN,NNP,Roman,Noun,1
7,Space,PROPN,NNP,Space,Noun,1
8,Telescope,PROPN,NNP,Telescope,Noun,1
9,Launches,PROPN,NNPS,Launches,Noun,1


### What is a *lemma*?

The `Lemma` column holds the **dictionary form** of a word. "Launched", "launches" and
"launching" all have the lemma "launch". This matters for counting: without lemmas,
those three would look like three unrelated words.

In [11]:
# A few rows that show lemmatisation doing something visible.
# .head(8) just limits it to the first 8 matches so the output stays readable.
changed = pos_df[pos_df["Word"].str.lower() != pos_df["Lemma"].str.lower()]

print("Words whose lemma differs from the original word:")
changed[["Word", "Lemma", "POS_Tag", "Fine_Tag"]].head(8)

Words whose lemma differs from the original word:


,Word,Lemma,POS_Tag,Fine_Tag
3,Seeking,seek,VERB,VBG
23,secrets,secret,NOUN,NNS
25,launched,launch,VERB,VBD
40,pairs,pair,VERB,VBZ
45,swaths,swath,NOUN,NNS
53,astronomers,astronomer,NOUN,NNS
57,worlds,world,NOUN,NNS
59,known,know,VERB,VBN


## Step 8 — What is this article actually about?

A useful side effect of POS tagging: the most frequent nouns tell you the subject
matter, and the most frequent verbs tell you what is happening.

In [12]:
# Count the most common nouns, using the lemma so that
# "telescope" and "telescopes" are counted together.
noun_rows = pos_df[pos_df["Category"] == "Noun"]
verb_rows = pos_df[pos_df["Category"] == "Verb"]

top_nouns = Counter(noun_rows["Lemma"].str.lower()).most_common(10)
top_verbs = Counter(verb_rows["Lemma"].str.lower()).most_common(10)

print("TOP 10 NOUNS")
print("-" * 30)
for word, count in top_nouns:
    # "#" * count draws a simple bar chart out of hash characters
    print(f"  {word:<15} {count:>3}  {'#' * count}")

print("\nTOP 10 VERBS")
print("-" * 30)
for word, count in top_verbs:
    print(f"  {word:<15} {count:>3}  {'#' * count}")

TOP 10 NOUNS
------------------------------
  roman            19  ###################
  nasa             14  ##############
  space            14  ##############
  mission           8  ########
  launch            8  ########
  telescope         7  #######
  universe          6  ######
  instrument        6  ######
  rocket            4  ####
  complex           4  ####

TOP 10 VERBS
------------------------------
  will             20  ####################
  be               11  ###########
  have              4  ####
  say               3  ###
  launch            2  ##
  pair              2  ##
  explore           2  ##
  help              2  ##
  know              2  ##
  see               2  ##


Read those two lists on their own and you can more or less reconstruct the story:
something to do with Roman, NASA, a telescope and a mission, which will *launch*,
*explore* and *survey*. That is POS tagging earning its keep.

In [13]:
# A compact summary table: how many of each tag, and what share of the total.
summary = (
    pos_df.groupby(["Category", "POS_Tag", "Fine_Tag"])
          .size()                             # count rows in each group
          .reset_index(name="Count")          # turn the result back into a table
          .sort_values(["Category", "Count"], ascending=[True, False])
)

# Add a percentage column, rounded to one decimal place.
summary["Percent"] = (100 * summary["Count"] / len(pos_df)).round(1)

print(f"Breakdown of all {len(pos_df)} extracted words:")
summary

Breakdown of all 483 extracted words:


,Category,POS_Tag,Fine_Tag,Count,Percent
0,Noun,NOUN,NN,156,32.3
2,Noun,PROPN,NNP,156,32.3
1,Noun,NOUN,NNS,58,12.0
3,Noun,PROPN,NNPS,3,0.6
9,Verb,VERB,VB,35,7.2
4,Verb,AUX,MD,22,4.6
11,Verb,VERB,VBG,11,2.3
10,Verb,VERB,VBD,9,1.9
12,Verb,VERB,VBN,9,1.9
14,Verb,VERB,VBZ,9,1.9


## Step 9 — Export to CSV

The required output filename is **`PrasannaKoirala_POS_01.csv`**, per the naming
convention in the brief.

In [14]:
# Make sure the outputs folder exists (creates it only if missing).
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH = OUTPUT_DIR / "PrasannaKoirala_POS_01.csv"

# index=False stops pandas writing its own row-number column into the file.
# encoding="utf-8" preserves any non-ASCII characters.
pos_df.to_csv(CSV_PATH, index=False, encoding="utf-8")

print("Saved:", CSV_PATH)
print("Rows :", len(pos_df))
print("Size :", CSV_PATH.stat().st_size, "bytes")

Saved: outputs/PrasannaKoirala_POS_01.csv
Rows : 483
Size : 15108 bytes


In [15]:
# Read the file back off disk and check it. Never trust a write you have not verified.
check = pd.read_csv(CSV_PATH)

print("Rows read back :", len(check))
print("Columns        :", list(check.columns))
print("Any empty cells:", check.isna().sum().sum())
print()
print("First 5 rows of the actual saved file:")
check.head()

Rows read back : 483
Columns        : ['Word', 'POS_Tag', 'Fine_Tag', 'Lemma', 'Category', 'Sentence_No']
Any empty cells: 0

First 5 rows of the actual saved file:


,Word,POS_Tag,Fine_Tag,Lemma,Category,Sentence_No
0,NASA,PROPN,NNP,NASA,Noun,1
1,Dark,PROPN,NNP,Dark,Noun,1
2,Universe,PROPN,NNP,Universe,Noun,1
3,Seeking,VERB,VBG,seek,Verb,1
4,Nancy,PROPN,NNP,Nancy,Noun,1


## Step 10 — Cross-check with a second tagger (NLTK)

The assignment only requires one tagger. We are running a second one anyway, because
comparing two taggers tells you something that running one never can: **where the
model is uncertain.**

NLTK is the older, more traditional NLP library. Its tagger uses a different algorithm
(an averaged perceptron) trained on different data, so where the two disagree is
usually where the language is genuinely ambiguous.

One important difference: **NLTK only produces the fine-grained Penn Treebank tags**,
so to compare like with like, we map its output onto the coarse categories ourselves.

In [16]:
import nltk
from nltk import word_tokenize, pos_tag

# NLTK needs its data files. They download once and are cached after that.
# quiet=True stops it printing progress bars.
for package in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(package, quiet=True)

# NLTK works in two separate steps (spaCy did both at once):
#   1. word_tokenize() splits the text into words
#   2. pos_tag()      assigns a tag to each word
nltk_tokens = word_tokenize(article_text)
nltk_tagged = pos_tag(nltk_tokens)

print("NLTK tokens :", len(nltk_tokens))
print("spaCy tokens:", len(doc))
print()
print("First 12 NLTK tags:")
print(nltk_tagged[:12])

NLTK tokens : 973
spaCy tokens: 1012

First 12 NLTK tags:
[('NASA', 'NNP'), ("'s", 'POS'), ('Dark', 'NNP'), ('Universe-Seeking', 'NNP'), ('Nancy', 'NNP'), ('Grace', 'NNP'), ('Roman', 'NNP'), ('Space', 'NNP'), ('Telescope', 'NNP'), ('Launches', 'NNP'), ('Now', 'RB'), ('on', 'IN')]


In [17]:
# NLTK gives only fine tags, so we write a small function to group them
# the same way spaCy's coarse tags do.
def coarse_category(penn_tag):
    """Turn a Penn Treebank tag into 'Noun', 'Verb', or None.

    Penn tags are prefix-based, which makes this easy:
      NN, NNS, NNP, NNPS all start with "NN"  -> noun
      VB, VBD, VBG, VBN, VBP, VBZ start "VB"  -> verb
      MD is the modal verb tag ("will", "could"), which spaCy calls AUX
    """
    if penn_tag.startswith("NN"):
        return "Noun"
    if penn_tag.startswith("VB") or penn_tag == "MD":
        return "Verb"
    return None

# Count what NLTK found.
nltk_counts = Counter(
    coarse_category(tag)
    for _, tag in nltk_tagged
    if coarse_category(tag) is not None
)

spacy_counts = Counter(pos_df["Category"])

print("SIDE BY SIDE")
print("=" * 44)
print(f"{'':<10} {'spaCy':>8} {'NLTK':>8} {'diff':>8}")
print("-" * 44)
for category in ["Noun", "Verb"]:
    s, n = spacy_counts[category], nltk_counts[category]
    print(f"{category:<10} {s:>8} {n:>8} {n - s:>+8}")

SIDE BY SIDE
              spaCy     NLTK     diff
--------------------------------------------
Noun            373      368       -5
Verb            110      107       -3


### First problem: the two tokenisers do not agree on what a word is

Before we can compare tags, we hit a snag. The two libraries split the text into a
**different number of tokens**, so we cannot simply walk both lists side by side.

In [18]:
# Drop spaCy's whitespace tokens so we are comparing like with like.
spacy_tokens = [t for t in doc if not t.is_space]

print("spaCy tokens:", len(spacy_tokens))
print("NLTK tokens :", len(nltk_tagged))
print("difference  :", len(spacy_tokens) - len(nltk_tagged))

# Find the first place where the two tokenisers produce a different word.
for i, (spacy_token, (nltk_word, _)) in enumerate(zip(spacy_tokens, nltk_tagged)):
    if spacy_token.text != nltk_word:
        print(f"\nFirst divergence at position {i}:")
        print("  spaCy split it as:", [t.text for t in spacy_tokens[i-1:i+4]])
        print("  NLTK  split it as:", [w for w, _ in nltk_tagged[i-1:i+3]])
        break

spaCy tokens: 998
NLTK tokens : 973
difference  : 25

First divergence at position 3:
  spaCy split it as: ['Dark', 'Universe', '-', 'Seeking', 'Nancy']
  NLTK  split it as: ['Dark', 'Universe-Seeking', 'Nancy', 'Grace']


There it is. The article contains the hyphenated phrase **"Dark Universe-Seeking"**:

- **spaCy** splits it into three tokens: `Universe` + `-` + `Seeking`
- **NLTK** keeps it as one token: `Universe-Seeking`

Neither is *wrong* — they are different conventions. But it means that from token 3
onward, the two lists are **offset from each other**. Position 50 in one list is not the
same word as position 50 in the other.

**So comparing them position by position would be meaningless.** It would report
disagreements everywhere, or nowhere, depending on how the offsets happened to fall —
and either way the number would be garbage.

### The fix: align the sequences first

Python's built-in `difflib` solves exactly this problem. `SequenceMatcher` finds the
**matching blocks** — runs of tokens that both lists agree on — and skips the parts
where they diverge. We then compare tags only inside those matching runs, which is a
genuine like-for-like comparison.

In [19]:
from difflib import SequenceMatcher

# Compare the two lists of WORDS (not tags) to find where they line up.
spacy_words = [t.text for t in spacy_tokens]
nltk_words = [word for word, tag in nltk_tagged]

# autojunk=False stops difflib from ignoring very common tokens as "noise",
# which matters here because words like "the" are exactly what we want matched.
matcher = SequenceMatcher(None, spacy_words, nltk_words, autojunk=False)

# get_matching_blocks() returns (a_start, b_start, size) triples meaning
# "spacy_words[a_start : a_start+size] equals nltk_words[b_start : b_start+size]".
aligned_pairs = []
for a_start, b_start, size in matcher.get_matching_blocks():
    for offset in range(size):
        spacy_token = spacy_tokens[a_start + offset]
        nltk_word, nltk_tag = nltk_tagged[b_start + offset]
        aligned_pairs.append((spacy_token, nltk_tag))

print(f"Tokens aligned successfully : {len(aligned_pairs)}")
print(f"Out of spaCy's              : {len(spacy_tokens)}")
print(f"Coverage                    : {len(aligned_pairs) / len(spacy_tokens):.1%}")

Tokens aligned successfully : 948
Out of spaCy's              : 998
Coverage                    : 95.0%


Compare that coverage figure to what naive side-by-side matching would have given —
about **1%**. Proper alignment is the difference between a real comparison and a
meaningless one.

In [20]:
# Now compare the tags, but ONLY on the properly aligned tokens.
disagreements = []
agreements = 0

for spacy_token, nltk_tag in aligned_pairs:
    if spacy_token.tag_ == nltk_tag:
        agreements += 1
    else:
        disagreements.append({
            "Word":       spacy_token.text,
            "spaCy_Tag":  spacy_token.tag_,
            "NLTK_Tag":   nltk_tag,
            "spaCy_says": spacy.explain(spacy_token.tag_),
            "NLTK_says":  spacy.explain(nltk_tag),
        })

total = agreements + len(disagreements)
print(f"Agree    : {agreements:>4} / {total}  ({agreements / total:.1%})")
print(f"Disagree : {len(disagreements):>4} / {total}  ({len(disagreements) / total:.1%})")

Agree    :  895 / 948  (94.4%)
Disagree :   53 / 948  (5.6%)


In [21]:
# What kinds of disagreement are there? Group them by the tag pair.
disagreement_df = pd.DataFrame(disagreements)

pattern_counts = (
    disagreement_df.groupby(["spaCy_Tag", "NLTK_Tag"])
                   .size()
                   .reset_index(name="Count")
                   .sort_values("Count", ascending=False)
)

print("MOST COMMON DISAGREEMENT PATTERNS")
pattern_counts.head(10)

MOST COMMON DISAGREEMENT PATTERNS


,spaCy_Tag,NLTK_Tag,Count
2,IN,TO,8
4,JJ,NNP,5
23,VB,NN,4
8,NN,NNP,3
18,RB,IN,3
12,NNP,JJ,3
17,NNS,NN,2
3,JJ,NN,2
16,NNPS,NNP,2
22,RP,IN,2


In [22]:
# And a sample of the actual words they disagree about.
disagreement_df.head(12)

,Word,spaCy_Tag,NLTK_Tag,spaCy_says,NLTK_says
0,Launches,NNPS,NNP,"noun, proper plural","noun, proper singular"
1,to,IN,TO,"conjunction, subordinating or preposition","infinitival ""to"""
2,launched,VBD,VBN,"verb, past tense","verb, past participle"
3,39A,NN,CD,"noun, singular or mass",cardinal number
4,Roman,JJ,NNP,"adjective (English), other noun-modifier (Chin...","noun, proper singular"
5,crisp,JJ,NN,"adjective (English), other noun-modifier (Chin...","noun, singular or mass"
6,infrared,JJ,VBN,"adjective (English), other noun-modifier (Chin...","verb, past participle"
7,probe,VB,NN,"verb, base form","noun, singular or mass"
8,Delivered,VBN,NNP,"verb, past participle","noun, proper singular"
9,atlas,NNS,NN,"noun, plural","noun, singular or mass"


### Reading the disagreements

Looking at the actual patterns, the disagreements split into **two very different
kinds** — and telling them apart matters.

#### Kind 1 — Tagset convention differences (not real disagreements)

The single biggest pattern is **`IN` vs `TO`**, and every instance is the word **"to"**.

- **NLTK** gives "to" its own dedicated tag, `TO`
- **spaCy** files it under `IN` (preposition/subordinating conjunction)

Neither is wrong. The two tagsets simply *define their categories differently*. This is
a bookkeeping difference, not a disagreement about what the word means. The same goes
for `RB` vs `IN` on words like "about" and "before".

**If you were measuring tagger accuracy, counting these as errors would be misleading.**

#### Kind 2 — Genuine linguistic ambiguity (the interesting ones)

Look at the **`VB` vs `NN`** disagreements. The words are:

> **image**, **power**, **probe**, **survey**

Every one of these can be *either* a noun or a verb, and you need the sentence to
decide:

- "Roman's Coronagraph will **image** Earth-like planets" → verb
- "The **image** was released" → noun
- "It will **survey** the universe" → verb
- "The **survey** covered the whole sky" → noun

**This is exactly the ambiguity described back in the introduction**, now showing up on
real data. The two models genuinely read these sentences differently.

Similarly, **`JJ` vs `NNP`** on the word **"Roman"** — in "the Roman telescope", is
"Roman" a proper noun (the mission's name) or an adjective modifying "telescope"? That
is a real question with no clean answer.

#### The takeaway

**This is why POS tagging accuracy is never 100%.** Some English is genuinely ambiguous,
and even trained human annotators disagree with each other on roughly 3% of tokens. A
single tagger hides that uncertainty completely; running two makes it visible.

For this assignment we use the **spaCy output** as our answer. It is the more modern
model, it gives us both tag levels, and its lemmas are better.

## Step 11 — Checking the work against the brief

Finally, verify every requirement the assignment listed.

In [23]:
# Programmatically confirm each requirement instead of just claiming it is done.
checks = {
    "Article loaded from a .txt file":
        ARTICLE_PATH.exists() and len(article_text) > 0,

    "POS tagging performed":
        len(doc) > 0 and all(t.pos_ for t in doc if not t.is_space),

    "Nouns extracted":
        (pos_df["Category"] == "Noun").sum() > 0,

    "Verbs extracted":
        (pos_df["Category"] == "Verb").sum() > 0,

    "CSV file created":
        CSV_PATH.exists(),

    "CSV has a Word column":
        "Word" in check.columns,

    "CSV has a POS_Tag column":
        "POS_Tag" in check.columns,

    "CSV row count matches the table":
        len(check) == len(pos_df),

    "No empty cells in the CSV":
        check.isna().sum().sum() == 0,

    "Filename follows the required convention":
        CSV_PATH.name == "PrasannaKoirala_POS_01.csv",
}

print("REQUIREMENT CHECKLIST")
print("=" * 52)
for requirement, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}]  {requirement}")

print("=" * 52)
print("ALL CHECKS PASSED" if all(checks.values()) else "SOMETHING FAILED - review above")

REQUIREMENT CHECKLIST
  [PASS]  Article loaded from a .txt file
  [PASS]  POS tagging performed
  [PASS]  Nouns extracted
  [PASS]  Verbs extracted
  [PASS]  CSV file created
  [PASS]  CSV has a Word column
  [PASS]  CSV has a POS_Tag column
  [PASS]  CSV row count matches the table
  [PASS]  No empty cells in the CSV
  [PASS]  Filename follows the required convention
ALL CHECKS PASSED


## Summary

**What was done**

1. Loaded an 850-word NASA news article from a plain `.txt` file
2. Tokenised and POS-tagged it with spaCy's `en_core_web_sm` model
3. Extracted every noun and verb, keeping both the coarse and fine-grained tags
4. Exported the result to `PrasannaKoirala_POS_01.csv`
5. Cross-checked the tagging against a second, independent tagger (NLTK)

**What was learned**

- POS tagging is context-dependent, not a dictionary lookup — "launch" is a noun or a
  verb depending on the sentence around it
- Tokenisation is a real step with real decisions (`NASA's` → `NASA` + `'s`), and
  everything downstream depends on getting it right
- Coarse and fine tagsets answer different questions, and it costs nothing to keep both
- "Extract all nouns" is ambiguous. Proper nouns turned out to be a large share of the
  nouns in this article, so whether you include them substantially changes the answer —
  a decision worth stating explicitly rather than making silently
- Two good taggers disagree on a small number of words, and where they disagree is
  usually where the English is genuinely ambiguous

**Files produced**

| File | Contents |
|---|---|
| `data/article.txt` | The original news article |
| `data/SOURCE.md` | Citation and why this article was chosen |
| `outputs/PrasannaKoirala_POS_01.csv` | Every extracted noun and verb |